# Adaptive Learning Rate — Theory Notebook

This notebook is the interactive companion to `notes.md`. It builds adaptive optimizers from scalar examples to multi-coordinate diagnostics using LaTeX-in-Markdown notation and executable NumPy code.

In [ ]:
import numpy as np
import numpy.linalg as la
import matplotlib as mpl

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

try:
    import seaborn as sns
    HAS_SNS = True
except ImportError:
    HAS_SNS = False

if HAS_MPL:
    if HAS_SNS:
        sns.set_theme(style="whitegrid", palette="colorblind")
    else:
        plt.style.use("seaborn-v0_8-whitegrid")
    mpl.rcParams.update({
        "figure.figsize": (10, 6),
        "figure.dpi": 120,
        "font.size": 13,
        "axes.titlesize": 15,
        "axes.labelsize": 13,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "legend.framealpha": 0.85,
        "lines.linewidth": 2.0,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "savefig.bbox": "tight",
        "savefig.dpi": 150,
    })

COLORS = {
    "primary": "#0077BB",
    "secondary": "#EE7733",
    "tertiary": "#009988",
    "error": "#CC3311",
    "neutral": "#555555",
    "highlight": "#EE3377",
}

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def check_close(name, got, expected, tol=1e-8):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} — {name}")
    if not ok:
        print("  expected:", expected)
        print("  got     :", got)
    return ok

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond

def adagrad_update(theta, g, r, lr=0.1, eps=1e-8):
    r = r + g * g
    step = lr * g / (np.sqrt(r) + eps)
    return theta - step, r, step

def rmsprop_update(theta, g, r, lr=0.1, rho=0.9, eps=1e-8):
    r = rho * r + (1 - rho) * (g * g)
    step = lr * g / (np.sqrt(r) + eps)
    return theta - step, r, step

def adam_update(theta, g, m, v, t, lr=0.001, b1=0.9, b2=0.999, eps=1e-8):
    m = b1 * m + (1 - b1) * g
    v = b2 * v + (1 - b2) * (g * g)
    mhat = m / (1 - b1 ** t)
    vhat = v / (1 - b2 ** t)
    step = lr * mhat / (np.sqrt(vhat) + eps)
    return theta - step, m, v, step, mhat, vhat

print("Setup complete.")


---

## 1.1 One global learning rate fails

Compare gradient descent on an ill-conditioned quadratic.

In [ ]:
# === 1.1 One global learning rate fails ===
H = np.diag([1.0, 100.0])
theta = np.array([4.0, 4.0])
lr = 0.018
trajectory = [theta.copy()]
for _ in range(80):
    g = H @ theta
    theta = theta - lr * g
    trajectory.append(theta.copy())
trajectory = np.array(trajectory)
loss = 0.5 * np.sum((trajectory @ H) * trajectory, axis=1)
print(f"initial loss: {loss[0]:.4f}")
print(f"final loss:   {loss[-1]:.4f}")
check_true("loss decreases despite zigzag", loss[-1] < loss[0])
if HAS_MPL:
    fig, ax = plt.subplots()
    ax.plot(trajectory[:, 0], label="$\\theta_1$", color=COLORS["primary"])
    ax.plot(trajectory[:, 1], label="$\\theta_2$", color=COLORS["secondary"])
    ax.set_title("One learning rate on an ill-conditioned quadratic")
    ax.set_xlabel("Step")
    ax.set_ylabel("Coordinate value")
    ax.legend()
    fig.tight_layout()
    plt.show()


---

## 1.2 Diagonal preconditioning

Rescale coordinates by a diagonal matrix.

In [ ]:
# === 1.2 Diagonal preconditioning ===
H = np.diag([1.0, 100.0])
theta_plain = np.array([4.0, 4.0])
theta_prec = np.array([4.0, 4.0])
P = np.diag([1.0, 0.01])
for _ in range(50):
    theta_plain -= 0.018 * (H @ theta_plain)
    theta_prec -= 0.2 * (P @ (H @ theta_prec))
loss_plain = 0.5 * theta_plain @ H @ theta_plain
loss_prec = 0.5 * theta_prec @ H @ theta_prec
print(f"plain GD loss:         {loss_plain:.6f}")
print(f"preconditioned loss:   {loss_prec:.6f}")
check_true("diagonal preconditioning improves this example", loss_prec < loss_plain)


---

## 1.3 Sparse gradients

Show how rare coordinates keep larger AdaGrad steps.

In [ ]:
# === 1.3 Sparse gradients ===
grads = np.zeros((20, 3))
grads[:, 0] = 1.0
grads[::5, 1] = 1.0
grads[10, 2] = 1.0
r = np.zeros(3)
eff = []
for g in grads:
    r += g * g
    eff.append(0.1 / (np.sqrt(r) + 1e-8))
eff = np.array(eff)
print("final effective LR:", eff[-1])
check_true("rare coordinate keeps largest effective LR", eff[-1, 2] > eff[-1, 0])


---

## 1.4 Optimizer lineage

Run the same gradient sequence through SGD, AdaGrad, RMSProp, and Adam.

In [ ]:
# === 1.4 Optimizer lineage ===
grads = np.array([1.0, 1.0, 1.0, 0.1, 0.1, 0.1])
theta_sgd = theta_ada = theta_rms = theta_adam = 1.0
r_ada = r_rms = m = v = 0.0
for t, g in enumerate(grads, 1):
    theta_sgd -= 0.1 * g
    theta_ada, r_ada, _ = adagrad_update(theta_ada, g, r_ada, lr=0.1)
    theta_rms, r_rms, _ = rmsprop_update(theta_rms, g, r_rms, lr=0.1)
    theta_adam, m, v, *_ = adam_update(theta_adam, g, m, v, t, lr=0.1)
print(f"SGD theta:     {theta_sgd:.6f}")
print(f"AdaGrad theta: {theta_ada:.6f}")
print(f"RMSProp theta: {theta_rms:.6f}")
print(f"Adam theta:    {theta_adam:.6f}")
check_true("methods produce different trajectories", len({round(x, 4) for x in [theta_sgd, theta_ada, theta_rms, theta_adam]}) > 2)


---

## 1.5 Transformer-style parameter groups

Inspect different synthetic gradient scales.

In [ ]:
# === 1.5 Transformer-style parameter groups ===
groups = {
    "embedding": np.random.normal(0, 0.03, size=1000),
    "attention": np.random.normal(0, 0.10, size=1000),
    "mlp": np.random.normal(0, 0.20, size=1000),
    "norm": np.random.normal(0, 0.005, size=1000),
}
for name, g in groups.items():
    print(f"{name:>10}: grad RMS={np.sqrt(np.mean(g*g)):.6f}")
check_true("MLP gradients are largest in this synthetic example", np.sqrt(np.mean(groups["mlp"]**2)) > np.sqrt(np.mean(groups["norm"]**2)))


---

## 2.1 Effective learning rate

Compute coordinatewise effective learning rates.

In [ ]:
# === 2.1 Effective learning rate ===
v = np.array([1e-4, 1e-2, 1.0, 100.0])
lr = 1e-3
eff = lr / (np.sqrt(v) + 1e-8)
print("v:", v)
print("effective LR:", eff)
check_true("larger second moment gives smaller effective LR", np.all(np.diff(eff) < 0))


---

## 2.2 Second moments

Compare cumulative and exponential squared-gradient statistics.

In [ ]:
# === 2.2 Second moments ===
g = np.r_[np.ones(8), 5*np.ones(8)]
cumulative = np.cumsum(g*g)
ema = []
r = 0.0
for x in g:
    r = 0.9 * r + 0.1 * x * x
    ema.append(r)
print(f"cumulative final: {cumulative[-1]:.3f}")
print(f"EMA final:        {ema[-1]:.3f}")
check_true("EMA adapts without growing unbounded", ema[-1] < cumulative[-1])


---

## 2.3 Preconditioner matrix

Build and apply a diagonal preconditioner.

In [ ]:
# === 2.3 Preconditioner matrix ===
g = np.array([10.0, 1.0, 0.1])
v = g * g
P = np.diag(1 / (np.sqrt(v) + 1e-8))
u = P @ g
print("raw gradient:", g)
print("preconditioned:", u)
check_close("absolute preconditioned components are one", np.abs(u), np.ones_like(u), tol=1e-6)


---

## 2.4 Bias correction

Verify Adam correction for constant gradients.

In [ ]:
# === 2.4 Bias correction ===
g = 3.0
b1 = 0.9
m = 0.0
for t in range(1, 6):
    m = b1 * m + (1 - b1) * g
    mhat = m / (1 - b1**t)
    print(f"t={t}, m={m:.6f}, mhat={mhat:.6f}")
check_close("bias-corrected moment recovers constant gradient", mhat, g)


---

## 2.5 State memory

Estimate optimizer memory by parameter count.

In [ ]:
# === 2.5 State memory ===
params = np.array([1e6, 1e8, 1e9, 1e10])
adam_state_gb = 2 * 4 * params / 1e9
for n, gb in zip(params, adam_state_gb):
    print(f"{n:>10.0f} params -> Adam state {gb:8.2f} GB")
check_close("1B params Adam state", adam_state_gb[2], 8.0)


---

## 3.1 AdaGrad sparse behavior

Rare coordinates retain step size.

In [ ]:
# === 3.1 AdaGrad sparse behavior ===
grads = np.array([[1, 0, 0], [1, 0, 0], [1, 1, 0], [1, 0, 0], [1, 0, 1]], dtype=float)
r = np.zeros(3)
for t, g in enumerate(grads, 1):
    r += g*g
    print(f"t={t}, accumulator={r}, eff_lr={0.1/(np.sqrt(r)+1e-8)}")
check_true("rare third coordinate has largest effective LR", (0.1/(np.sqrt(r)+1e-8))[2] > (0.1/(np.sqrt(r)+1e-8))[0])


---

## 3.2 AdaGrad diminishing steps

Persistent gradients shrink effective LR.

In [ ]:
# === 3.2 AdaGrad diminishing steps ===
r = 0.0
steps = []
for _ in range(100):
    r += 1.0
    steps.append(0.1 / np.sqrt(r))
print(f"first effective LR: {steps[0]:.6f}")
print(f"last effective LR:  {steps[-1]:.6f}")
check_true("AdaGrad effective LR diminishes", steps[-1] < steps[0])


---

## 3.3 RMSProp forgetting

A gradient scale shift is tracked with finite memory.

In [ ]:
# === 3.3 RMSProp forgetting ===
g = np.r_[np.ones(20), 10*np.ones(20), np.ones(20)]
r = 0.0
track = []
for x in g:
    r = 0.9*r + 0.1*x*x
    track.append(r)
print(f"before spike: {track[19]:.3f}, after spike: {track[39]:.3f}, after recovery: {track[-1]:.3f}")
check_true("RMSProp forgets old spike partly", track[-1] < track[39])


---

## 3.4 Epsilon sensitivity

Observe maximum effective LR as epsilon changes.

In [ ]:
# === 3.4 Epsilon sensitivity ===
v = 1e-12
for eps in [1e-8, 1e-6, 1e-4, 1e-2]:
    eff = 1e-3 / (np.sqrt(v) + eps)
    print(f"epsilon={eps:g}, effective LR={eff:.6f}")
check_true("larger epsilon caps effective LR", 1e-3/(np.sqrt(v)+1e-2) < 1e-3/(np.sqrt(v)+1e-8))


---

## 3.5 AdaDelta intuition

Track gradient and update RMS values.

In [ ]:
# === 3.5 AdaDelta intuition ===
g2 = 0.0
u2 = 0.0
rho = 0.95
for t in range(1, 6):
    g = 1.0 / t
    g2 = rho*g2 + (1-rho)*g*g
    update = -np.sqrt(u2 + 1e-6) / np.sqrt(g2 + 1e-6) * g
    u2 = rho*u2 + (1-rho)*update*update
    print(f"t={t}, grad_rms={np.sqrt(g2):.6f}, update={update:.6f}, update_rms={np.sqrt(u2):.6f}")
check_true("AdaDelta-style update is finite", np.isfinite(update))


---

## 4.1 Adam components

Separate momentum and second-moment normalization.

In [ ]:
# === 4.1 Adam components ===
theta = 1.0
m = 0.0
v = 0.0
for t, g in enumerate([2.0, 2.0, 2.0], 1):
    theta, m, v, step, mhat, vhat = adam_update(theta, g, m, v, t, lr=0.1)
    print(f"t={t}, mhat={mhat:.6f}, vhat={vhat:.6f}, step={step:.6f}, theta={theta:.6f}")
check_true("Adam moved parameter downward", theta < 1.0)


---

## 4.2 Bias correction curves

Plot correction factors over time.

In [ ]:
# === 4.2 Bias correction curves ===
t = np.arange(1, 51)
c1 = 1 - 0.9**t
c2 = 1 - 0.999**t
print(f"first c1={c1[0]:.6f}, c2={c2[0]:.6f}")
print(f"step 50 c1={c1[-1]:.6f}, c2={c2[-1]:.6f}")
check_true("second moment correction grows slowly", c2[-1] < c1[-1])
if HAS_MPL:
    fig, ax = plt.subplots()
    ax.plot(t, c1, label="$1-\\beta_1^t$", color=COLORS["primary"])
    ax.plot(t, c2, label="$1-\\beta_2^t$", color=COLORS["secondary"])
    ax.set_title("Adam bias-correction factors")
    ax.set_xlabel("Step $t$")
    ax.set_ylabel("Correction denominator")
    ax.legend()
    fig.tight_layout()
    plt.show()


---

## 4.3 Adam sign-like regime

Constant gradients produce normalized updates.

In [ ]:
# === 4.3 Adam sign-like regime ===
for g in [0.01, 1.0, 100.0]:
    step = 0.001 * g / (np.sqrt(g*g) + 1e-8)
    print(f"g={g:8.3g}, normalized Adam-like step={step:.6f}")
check_close("large and small positive gradients normalize similarly", 0.001*0.01/(0.01+1e-8), 0.001*100/(100+1e-8), tol=1e-6)


---

## 4.4 AMSGrad maximum

Keep nondecreasing denominators.

In [ ]:
# === 4.4 AMSGrad maximum ===
vhat = np.array([4.0, 1.0, 9.0, 2.0, 3.0])
vmax = np.maximum.accumulate(vhat)
print("vhat: ", vhat)
print("vmax: ", vmax)
check_true("AMSGrad denominator is nondecreasing", np.all(np.diff(vmax) >= 0))


---

## 4.5 AdamW versus L2

Show coupled and decoupled decay differ.

In [ ]:
# === 4.5 AdamW versus L2 ===
theta = np.array([1.0, 10.0])
g = np.array([0.1, 0.1])
v = np.array([1e-4, 1.0])
lr = 0.01
wd = 0.1
coupled = theta - lr * (g + wd*theta) / (np.sqrt(v) + 1e-8)
decoupled = (1 - lr*wd) * theta - lr * g / (np.sqrt(v) + 1e-8)
print("coupled Adam-style L2:", coupled)
print("decoupled AdamW:      ", decoupled)
check_true("coupled and decoupled decay differ", not np.allclose(coupled, decoupled))


---

## 5.1 LARS and LAMB

Compute trust ratios.

In [ ]:
# === 5.1 LARS and LAMB ===
theta = np.array([3.0, 4.0])
u = np.array([0.06, 0.08])
trust = la.norm(theta) / (la.norm(u) + 1e-8)
print(f"parameter norm: {la.norm(theta):.6f}")
print(f"update norm:    {la.norm(u):.6f}")
print(f"trust ratio:    {trust:.6f}")
check_close("trust ratio", trust, 50.0, tol=1e-5)


---

## 5.2 Layerwise scaling

Compare update-to-parameter ratios.

In [ ]:
# === 5.2 Layerwise scaling ===
param_norms = np.array([10.0, 1.0, 0.1])
update_norms = np.array([1.0, 1.0, 1.0])
trust = param_norms / (update_norms + 1e-8)
print("trust ratios:", trust)
check_true("small-parameter layer gets smallest trust ratio", trust[-1] < trust[0])


---

## 5.3 Adafactor factoring

Approximate a second-moment matrix from row and column stats.

In [ ]:
# === 5.3 Adafactor factoring ===
V = np.abs(np.random.lognormal(mean=0.0, sigma=0.6, size=(4, 6)))
r = V.mean(axis=1)
c = V.mean(axis=0)
Vhat = np.outer(r, c) / r.mean()
rel_err = la.norm(Vhat - V) / la.norm(V)
print(f"full entries: {V.size}, factored entries: {len(r)+len(c)}")
print(f"relative reconstruction error: {rel_err:.6f}")
check_true("factored state uses fewer entries", len(r)+len(c) < V.size)


---

## 5.4 State memory at scale

Plot memory costs for AdamW and Adafactor-style factoring.

In [ ]:
# === 5.4 State memory at scale ===
params = np.logspace(6, 11, 6)
adam_gb = 2 * 4 * params / 1e9
for n, gb in zip(params, adam_gb):
    print(f"{n:>12.0f} params -> AdamW state {gb:8.2f} GB")
check_true("100B parameter Adam state is enormous", adam_gb[-1] > 500)


---

## 5.5 When layerwise scaling hurts

Tiny parameter norms can create unstable trust ratios.

In [ ]:
# === 5.5 When layerwise scaling hurts ===
param_norm = 1e-6
update_norm = 1e-2
trust = param_norm / (update_norm + 1e-8)
print(f"tiny layer trust ratio: {trust:.8f}")
check_true("trust ratio can nearly freeze tiny-norm layers", trust < 1e-3)


---

## 6.1 Diagonal natural-gradient bridge

Compare diagonal Fisher-style scaling.

In [ ]:
# === 6.1 Diagonal natural-gradient bridge ===
g = np.array([2.0, 0.5])
diag_fisher = np.array([4.0, 0.25])
nat_diag_step = g / (diag_fisher + 1e-8)
adam_like_step = g / (np.sqrt(diag_fisher) + 1e-8)
print("diagonal Fisher inverse step:", nat_diag_step)
print("Adam-like inverse-root step: ", adam_like_step)
check_true("both rescale by geometry but differently", not np.allclose(nat_diag_step, adam_like_step))


---

## 6.2 Curvature limits

Show diagonal scaling cannot remove rotation.

In [ ]:
# === 6.2 Curvature limits ===
angle = np.pi / 4
Q = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
H = Q @ np.diag([1.0, 100.0]) @ Q.T
diag_precond = np.diag(1 / np.diag(H))
residual = diag_precond @ H
print("preconditioned Hessian:")
print(residual)
print(f"off-diagonal magnitude: {abs(residual[0,1]):.6f}")
check_true("diagonal scaling leaves off-diagonal coupling", abs(residual[0,1]) > 0.1)


---

## 6.3 Sign-based updates

Compare Adam-like normalized update to sign update.

In [ ]:
# === 6.3 Sign-based updates ===
g = np.array([-10.0, -0.1, 0.2, 5.0])
adam_like = g / (np.sqrt(g*g) + 1e-8)
sign_update = np.sign(g)
print("Adam-like normalized:", adam_like)
print("sign update:         ", sign_update)
check_close("normalization approaches sign", adam_like, sign_update, tol=1e-6)


---

## 6.4 Structured preconditioner preview

Apply a left/right matrix preconditioner.

In [ ]:
# === 6.4 Structured preconditioner preview ===
G = np.array([[3.0, 1.0], [1.0, 2.0]])
L_inv_quarter = np.diag([0.5, 1.0])
R_inv_quarter = np.diag([1.0, 0.25])
U = L_inv_quarter @ G @ R_inv_quarter
print("raw gradient matrix:")
print(G)
print("structured preconditioned update:")
print(U)
check_true("structured preconditioning changes matrix anisotropically", not np.allclose(G, U))


---

## 6.5 Stability diagnostics

Compute gradient, update, parameter, and ratio logs.

In [ ]:
# === 6.5 Stability diagnostics ===
theta = np.random.normal(size=100)
g = np.random.normal(scale=0.1, size=100)
v = np.square(g) + 1e-4
update = -1e-3 * g / (np.sqrt(v) + 1e-8)
ratio = la.norm(update) / la.norm(theta)
print(f"grad norm:      {la.norm(g):.6f}")
print(f"update norm:    {la.norm(update):.6f}")
print(f"parameter norm: {la.norm(theta):.6f}")
print(f"update/param:   {ratio:.8f}")
check_true("ratio is finite and positive", np.isfinite(ratio) and ratio > 0)


---

## 7.1 AdamW for transformers

Synthetic parameter groups with no-decay rules.

In [ ]:
# === 7.1 AdamW for transformers ===
params = {"q_proj.weight": True, "mlp.weight": True, "norm.weight": False, "bias": False}
for name, decay in params.items():
    print(f"{name:>14}: {'decay' if decay else 'no_decay'}")
check_true("norm parameters are excluded from decay", params["norm.weight"] is False)


---

## 7.2 Sparse embeddings

Per-row AdaGrad accumulators.

In [ ]:
# === 7.2 Sparse embeddings ===
rows = 5
acc = np.zeros(rows)
accesses = [0, 0, 0, 1, 0, 2, 0, 3, 4, 0]
for row in accesses:
    acc[row] += 1.0
eff = 0.1 / (np.sqrt(acc) + 1e-8)
print("row accumulators:", acc)
print("row effective LR:", eff)
check_true("rare row has larger LR than common row", eff[4] > eff[0])


---

## 7.3 Reinforcement learning noise

EMA smoothing for nonstationary gradients.

In [ ]:
# === 7.3 Reinforcement learning noise ===
signal = np.r_[np.ones(20), -np.ones(20)]
noise = np.random.normal(scale=2.0, size=40)
g = signal + noise
m = 0.0
ema = []
for x in g:
    m = 0.9*m + 0.1*x
    ema.append(m)
print(f"raw gradient std: {np.std(g):.6f}")
print(f"EMA gradient std: {np.std(ema):.6f}")
check_true("EMA smooths noisy gradients", np.std(ema) < np.std(g))


---

## 7.4 Large-batch LAMB

Trust ratio with batch-dependent update scale.

In [ ]:
# === 7.4 Large-batch LAMB ===
theta_norm = 20.0
for update_norm in [0.5, 1.0, 2.0, 4.0]:
    trust = theta_norm / update_norm
    print(f"update_norm={update_norm:.1f}, trust={trust:.2f}, scaled_update_norm={trust*update_norm:.2f}")
check_close("trust ratio normalizes update to parameter norm", theta_norm, 20.0)


---

## 7.5 Optimizer checklist

Score candidate optimizers for workloads.

In [ ]:
# === 7.5 Optimizer checklist ===
workloads = ["transformer", "sparse_embeddings", "vision", "large_batch"]
recommend = {"transformer": "AdamW", "sparse_embeddings": "AdaGrad/Adam", "vision": "SGD+momentum or AdamW", "large_batch": "AdamW then LAMB"}
for w in workloads:
    print(f"{w:>18}: {recommend[w]}")
check_true("transformer baseline is AdamW", recommend["transformer"] == "AdamW")


---

## 8 Common mistakes

Numerically demonstrate two common myths.

In [ ]:
# === 8 Common mistakes ===
eta = 1e-3
v_small = 1e-12
eff_tiny_eps = eta / (np.sqrt(v_small) + 1e-8)
eff_large_eps = eta / (np.sqrt(v_small) + 1e-3)
print(f"effective LR with tiny epsilon:  {eff_tiny_eps:.3f}")
print(f"effective LR with larger epsilon:{eff_large_eps:.3f}")
check_true("epsilon is not just cosmetic", eff_tiny_eps > 10 * eff_large_eps)


---

## 9 Exercises preview

List the practice arc.

In [ ]:
# === 9 Exercises preview ===
topics = ["AdaGrad", "RMSProp", "Adam bias correction", "AdamW", "LAMB", "Adafactor", "Diagnostics"]
for i, topic in enumerate(topics, 1):
    print(f"{i}. {topic}")
check_true("practice arc includes AdamW", "AdamW" in topics)


---

## 10 2026 AI perspective

Compute optimizer state for model scales.

In [ ]:
# === 10 2026 AI perspective ===
params_b = np.array([1, 7, 70, 405])
adam_state_gb = params_b * 1e9 * 8 / 1e9
for b, gb in zip(params_b, adam_state_gb):
    print(f"{b:>4.0f}B params -> Adam state about {gb:>7.1f} GB")
check_true("frontier-scale optimizer state dominates memory", adam_state_gb[-1] > 1000)


---

## 11 Conceptual bridge

Summarize adaptation versus schedule and regularization.

In [ ]:
# === 11 Conceptual bridge ===
print("11 Conceptual bridge")
check_true("section executed", True)
